In [5]:
import pandas as pd

import ast

In [6]:
DATE = '2024-08-19'
eval_results_df = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_evaluation_results_{DATE}_diff_thresholds.csv")
jq_error_analysis_df = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_prediction_errors_{DATE}_diff_thresholds.csv")

eval_results_df['True'] = eval_results_df['True'].apply(lambda x: ast.literal_eval(x))

## Show overall metrics per JQ measure

In [7]:
metrics = []
for i, row in eval_results_df.iterrows():
    m = {'jq_measure': row['JQ_measure_name']}
    m.update({k:round(v,3) for k,v in row['True'].items()})
    metrics.append(m)
metrics = pd.DataFrame(metrics)
metrics

,jq_measure,precision,recall,f1-score,support
0,L&D,0.875,0.875,0.875,48.0
1,CAREER,0.900,0.720,0.800,25.0
2,HOURS,0.906,0.983,0.943,59.0
3,FLEX_HOURS,0.732,0.857,0.789,35.0
4,SHIFT,0.714,0.294,0.417,17.0
5,LOC,0.615,0.211,0.314,38.0
6,FLEX_LOC,0.842,0.800,0.821,20.0
7,CONTRACT,0.757,0.718,0.737,39.0
8,LEAVE,0.853,0.967,0.906,30.0
9,COMP,1.000,0.860,0.925,86.0


In [8]:
for i, row in eval_results_df.iterrows():
    print(f"|{row['JQ_measure_name']}|{round(row['True']['support'])}|")

|L&D|48|
|CAREER|25|
|HOURS|59|
|FLEX_HOURS|35|
|SHIFT|17|
|LOC|38|
|FLEX_LOC|20|
|CONTRACT|39|
|LEAVE|30|
|COMP|86|
|PERKS|55|
|CARING|8|
|DISABILITY|2|
|HEALTH|7|
|M_HEALTH|4|
|SPONSORSHIP|2|
|REWARD|3|
|MISC|13|
|AUTONOMY|1|
|SENSE OF PURPOSE|4|
|SOCIAL|22|
|VOICE REPRESENTATION|0|


## Deeper dive
1. Examples of when it does badly for each JQ measure
2. Parent sectors that are better and worse
3. Match threshold and metrics

In [9]:
jq_measure = 'L&D'
error_type = 'FN' # 'FN', 'FP'
for i, row in jq_error_analysis_df[jq_error_analysis_df[jq_measure]==error_type].iterrows():
    print('---')
    print({k:v for k,v in ast.literal_eval(row['jq_sentences']).items() if jq_measure in v})
    print([v for v in ast.literal_eval(row['pred_ngram_matched']) if v[3] == jq_measure] )

---
{"At Tradewind you will have access to 25 fully certified CPD courses, that's 18 more than our next nearest competitor,  all focused on making you the best you can be.": ['L&D'], 'We care about your training and development more than any other agency - which is why we can offer you more certified CPD courses than any other education recruitment agency, 25 to be exact!': ['L&D']}
[]
---
{'· Can confidently read Engineering drawings Keywords  CNC, Miller, Milling, Machinist, Lathe, Turning, Setting, Operating, 3-axis, axis, Mil, Contract, Temporary, Permanent, Engineering, Technical, Technician, CNC, Machining, Machinist, Setting, Setter, Operating, Operator, Manufacturing, Production,  Cambridgeshire, Miller, Milling, Double Days, progression, training, Milling.': ['L&D']}
[]
---
{'You should be able to demonstrate ambition to succeed and will in turn be provided with the tools and training to fulfil that desire.': ['L&D']}
[]
---
{'An understanding of the basic principles of Revenu

In [10]:
jq_error_analysis_df.groupby('parent_sector')['n_incorrect'].mean()

parent_sector
Accountancy                    2.000000
Accountancy (Qualified)        2.166667
Admin, Secretarial &amp; PA    1.571429
Banking                        0.000000
Charity &amp; Voluntary        2.000000
Construction &amp; Property    2.500000
Education                      2.666667
Energy                         2.000000
Engineering                    4.500000
Estate Agency                  2.000000
FMCG                           1.000000
Health &amp; Medicine          2.500000
Hospitality &amp; Catering     2.555556
Human Resources                1.500000
IT &amp; Telecoms              1.333333
Marketing &amp; PR             2.333333
Motoring &amp; Automotive      3.500000
Recruitment Consultancy        4.000000
Retail                         1.875000
Sales                          3.250000
Social Care                    3.500000
Transport &amp; Logistics      1.714286
Name: n_incorrect, dtype: float64

## Threshold

In [99]:
DATE = '2024-08-19'
jq_error_analysis_df_no_thresh = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_prediction_errors_{DATE}_no_thresh.csv")

In [91]:
from sklearn.metrics import (
    classification_report,
)
import altair as alt

In [92]:
jq_cols = [
    "L&D",
    "CAREER",
    "HOURS",
    "FLEX_HOURS",
    "SHIFT",
    "LOC",
    "FLEX_LOC",
    "CONTRACT",
    "LEAVE",
    "COMP",
    "PERKS",
    "CARING",
    "DISABILITY",
    "HEALTH",
    "M_HEALTH",
    "SPONSORSHIP",
    "REWARD",
    "MISC",
    "AUTONOMY",
    "SENSE OF PURPOSE",
    "SOCIAL",
    "VOICE REPRESENTATION",
]

In [100]:
def get_thresh_result(x, cs_threshold, jq_measure):
    if pd.notnull(x):
        x = ast.literal_eval(x)
        res = [cat for _, thresh, _, cat in x if ((thresh>=cs_threshold) & (cat==jq_measure))]
        if len(res) ==0:
            return False
        else:
            return True
    else:
        return False
        


In [105]:
class_rep_per_jq = []
for jq_measure in jq_cols:
    truth_list = jq_error_analysis_df_no_thresh[jq_measure].isin(["TP", "FN"]).tolist()
    for cs_threshold in [0.3,0.35,0.4,0.45,0.5,0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
        pred_list = jq_error_analysis_df_no_thresh['pred_ngram_matched'].apply(lambda x: get_thresh_result(x, cs_threshold, jq_measure))
        class_rep = classification_report(
            truth_list, pred_list, output_dict=True
        )
        if 'True' not in class_rep:
            class_rep['True'] = {'recall': None, 'precision': None, 'f1-score': None, 'support': None}
        class_rep_per_jq.append(
            {'jq_measure': jq_measure,
             'threshold': cs_threshold,
             'recall': class_rep['True']['recall'],
             'precision': class_rep['True']['precision'],
             'f1-score': class_rep['True']['f1-score'],
             'support': class_rep['True']['support'],
            })

class_rep_per_jq_df = pd.DataFrame(class_rep_per_jq)
class_rep_per_jq_df

/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to cont

,jq_measure,threshold,recall,precision,f1-score,support
0,L&D,0.30,0.937500,0.600000,0.731707,48.0
1,L&D,0.35,0.937500,0.608108,0.737705,48.0
2,L&D,0.40,0.937500,0.652174,0.769231,48.0
3,L&D,0.45,0.916667,0.687500,0.785714,48.0
4,L&D,0.50,0.895833,0.796296,0.843137,48.0
...,...,...,...,...,...,...
303,VOICE REPRESENTATION,0.75,NaN,NaN,NaN,NaN
304,VOICE REPRESENTATION,0.80,NaN,NaN,NaN,NaN
305,VOICE REPRESENTATION,0.85,NaN,NaN,NaN,NaN
306,VOICE REPRESENTATION,0.90,NaN,NaN,NaN,NaN


In [106]:
jq_measures = class_rep_per_jq_df[((class_rep_per_jq_df['threshold']==0.55) & (class_rep_per_jq_df['support']>10))]['jq_measure'].tolist()
jq_measures = [j for j in jq_measures if j not in ['MISC', 'SOCIAL']]
filtered_jq_data = class_rep_per_jq_df[class_rep_per_jq_df['jq_measure'].isin(jq_measures[0:5])]
filtered_jq_data2 = class_rep_per_jq_df[class_rep_per_jq_df['jq_measure'].isin(jq_measures[5:])]

rec_plot = alt.Chart(filtered_jq_data, title="Recall (dashed), precision (bold)").mark_line(strokeDash=[1,1], strokeWidth=5).encode(
    x='threshold', y='recall', color='jq_measure', 
)
prec_plot = alt.Chart(filtered_jq_data).mark_line(strokeWidth=5).encode(
    x='threshold', y='precision', color='jq_measure'
)

rec_plot2 = alt.Chart(filtered_jq_data2, title="Recall (dashed), precision (bold)").mark_line(strokeDash=[1,1], strokeWidth=5).encode(
    x='threshold', y='recall', color='jq_measure', 
)
prec_plot2 = alt.Chart(filtered_jq_data2).mark_line(strokeWidth=5).encode(
    x='threshold', y='precision', color='jq_measure'
)

((rec_plot+prec_plot) | (rec_plot2+prec_plot2)).resolve_scale(color='independent')

alt.HConcatChart(...)

Changes to threshold that would make results more precise and not effect recall:
- CAREER - 0.6 
- FLEX_HOURS - 0.65
- HOURS - 0.6
- FLEX_LOC - 0.6
- LEAVE - 0.65